<a href="https://colab.research.google.com/github/anshpunia639-bit/Python/blob/main/Ansh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
PROJECT 1: Retail Store Sales Analysis

Assumed columns (rename in the CONFIG section if yours differ):
    transaction_id, date, branch, category, product,
    customer_id, quantity, unit_price, discount (% e.g. 10 = 10%)
"total_amount" is calculated if it is missing.
"""

import numpy as np
import pandas as pd


FILE_PATH = "sales.csv"      # your dataset path
COL = {
    "id": "transaction_id",
    "date": "date",
    "branch": "branch",
    "category": "category",
    "product": "product",
    "customer": "customer_id",
    "qty": "quantity",
    "price": "unit_price",
    "discount": "discount",
}

# ------------------------------------------------------------------
# LOAD DATA (falls back to sample data so the code always runs)
# ------------------------------------------------------------------
def load_data():
    try:
        df = pd.read_csv(FILE_PATH)
        df = df.rename(columns={v: k for k, v in COL.items()})
        print(f"Loaded {FILE_PATH}")
    except FileNotFoundError:
        print("sales.csv not found -> using generated sample data")
        rng = np.random.default_rng(42)
        n = 2000
        cats = {"Grocery": ["Rice", "Milk", "Bread"],
                "Electronics": ["Phone", "Headset", "Charger"],
                "Clothing": ["Shirt", "Jeans", "Jacket"],
                "Home": ["Lamp", "Pillow", "Kettle"]}
        category = rng.choice(list(cats), n)
        df = pd.DataFrame({
            "id": np.arange(1, n + 1),
            "date": pd.to_datetime("2024-01-01")
                    + pd.to_timedelta(rng.integers(0, 365, n), unit="D"),
            "branch": rng.choice(["North", "South", "East", "West"], n),
            "category": category,
            "product": [rng.choice(cats[c]) for c in category],
            "customer": rng.integers(1, 300, n),
            "qty": rng.integers(1, 10, n),
            "price": rng.uniform(5, 500, n).round(2),
            "discount": rng.choice([0, 5, 10, 15, 20, 30], n),
        })
    return df


def clean_data(df):
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date", "qty", "price"]).drop_duplicates()
    df["discount"] = df["discount"].fillna(0)
    if "total_amount" not in df.columns:
        df["total_amount"] = df["qty"] * df["price"] * (1 - df["discount"] / 100)
    df["month"] = df["date"].dt.to_period("M")
    df["month_name"] = df["date"].dt.month_name()
    df["weekday"] = df["date"].dt.day_name()
    df["quarter"] = df["date"].dt.to_period("Q")
    return df


def header(text):
    print("\n" + "=" * 70)
    print(text)
    print("=" * 70)


df = clean_data(load_data())
print(df.head())
print(df.info())


header("Q1. Overall sales & trend over time")
total_sales = df["total_amount"].sum()
print(f"Total sales value : {total_sales:,.2f}")
print(f"Total transactions: {len(df):,}")

monthly = df.groupby("month")["total_amount"].sum().to_frame("sales")
monthly["growth_%"] = monthly["sales"].pct_change() * 100
print(monthly.round(2))


monthly["moving_avg_3m"] = monthly["sales"].rolling(3).mean()
print("\nBest month :", monthly["sales"].idxmax())
print("Worst month:", monthly["sales"].idxmin())


header("Q2. Category & product revenue share")
cat_rev = df.groupby("category")["total_amount"].sum().sort_values(ascending=False)
cat_share = (cat_rev / cat_rev.sum() * 100).round(2)
print(pd.DataFrame({"revenue": cat_rev.round(2), "share_%": cat_share}))

prod_rev = df.groupby("product")["total_amount"].sum().sort_values(ascending=False)
print("\nTop 5 products:\n", prod_rev.head(5).round(2))
print("\nBottom 5 products:\n", prod_rev.tail(5).round(2))


header("Q3. Branch performance")
branch = df.groupby("branch").agg(
    revenue=("total_amount", "sum"),
    transactions=("id", "count"),
    avg_transaction=("total_amount", "mean"),
).sort_values("revenue", ascending=False)
branch["share_%"] = branch["revenue"] / branch["revenue"].sum() * 100
print(branch.round(2))


header("Q4. Best / weakest months, weekdays, quarters")
branch_month = df.pivot_table(index="branch", columns="month_name",
                              values="total_amount", aggfunc="sum")
print("Branch x Month sales:\n", branch_month.round(0))

weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday",
                 "Friday", "Saturday", "Sunday"]
weekday = df.groupby("weekday")["total_amount"].sum().reindex(weekday_order)
print("\nSales by weekday:\n", weekday.round(2))
print("\nSales by quarter:\n", df.groupby("quarter")["total_amount"].sum().round(2))


bm = df.groupby(["branch", "month"])["total_amount"].sum().reset_index()
stats = bm.groupby("branch")["total_amount"].transform
bm["z_score"] = (bm["total_amount"] - stats("mean")) / stats("std")
print("\nBranch-months needing investigation (z < -1):\n",
      bm[bm["z_score"] < -1].round(2))


header("Q5. Average transaction value")
atv_cat = df.groupby("category")["total_amount"].mean().round(2)
atv_branch = df.groupby("branch")["total_amount"].mean().round(2)
print("By category:\n", atv_cat)
print("\nBy branch:\n", atv_branch)
print("\nCategory x Branch:\n",
      df.pivot_table(index="category", columns="branch",
                     values="total_amount", aggfunc="mean").round(2))


header("Q6. Discount vs quantity / transaction value")
print("Correlations:\n",
      df[["discount", "qty", "total_amount"]].corr().round(3))

df["discount_band"] = pd.cut(df["discount"], bins=[-1, 0, 10, 20, 100],
                             labels=["None", "1-10%", "11-20%", ">20%"])
print("\nBy discount band:\n",
      df.groupby("discount_band", observed=True).agg(
          avg_qty=("qty", "mean"),
          avg_transaction=("total_amount", "mean"),
          transactions=("id", "count")).round(2))

print("\nAvg discount by category:\n", df.groupby("category")["discount"].mean().round(2))
print("\nAvg discount by branch:\n", df.groupby("branch")["discount"].mean().round(2))


header("Q7. Frequent & high-value customers")
cust = df.groupby("customer").agg(
    purchases=("id", "count"),
    total_spent=("total_amount", "sum"),
    avg_spent=("total_amount", "mean"),
)
print("Top 10 by spend:\n", cust.sort_values("total_spent", ascending=False).head(10).round(2))
print("\nTop 10 by frequency:\n", cust.sort_values("purchases", ascending=False).head(10).round(2))


freq_cut = np.percentile(cust["purchases"], 75)
value_cut = np.percentile(cust["total_spent"], 75)
cust["segment"] = np.select(
    [(cust["purchases"] >= freq_cut) & (cust["total_spent"] >= value_cut),
     cust["total_spent"] >= value_cut,
     cust["purchases"] >= freq_cut],
    ["Champion", "High-value", "Frequent"],
    default="Regular",
)
print("\nSegments:\n", cust["segment"].value_counts())


header("Q8. Recommended business actions")
top_cat, low_cat = cat_rev.idxmax(), cat_rev.idxmin()
top_branch, low_branch = branch.index[0], branch.index[-1]
best_day, worst_day = weekday.idxmax(), weekday.idxmin()
disc_corr = df["discount"].corr(df["qty"])

actions = [
    f"Stock more of '{top_cat}' (top category); review range/pricing for '{low_cat}'.",
    f"Replicate '{top_branch}' practices at '{low_branch}' (lowest revenue branch).",
    f"Run promotions on '{worst_day}' (weakest day); keep stock high for '{best_day}'.",
    f"Discount-quantity correlation is {disc_corr:.2f}: "
    + ("discounts barely move volume -> use bundles/loyalty offers instead."
       if abs(disc_corr) < 0.3 else "discounts do drive volume -> use targeted, limited ones."),
    f"Reward the {int((cust['segment'] == 'Champion').sum())} 'Champion' customers with loyalty perks.",
    "Cross-sell: bundle top products with slow-moving ones to raise average transaction value.",
]
for i, a in enumerate(actions, 1):
    print(f"{i}. {a}")


sales.csv not found -> using generated sample data
   id       date branch     category  product  customer  qty   price  \
0   1 2024-11-16  South      Grocery     Milk       194    4  230.46   
1   2 2024-01-23   West         Home   Kettle       215    2   63.83   
2   3 2024-05-25  North     Clothing    Jeans        80    5  286.74   
3   4 2024-06-16  South  Electronics  Headset       107    6   82.43   
4   5 2024-10-18   West  Electronics  Headset       283    3   35.17   

   discount  total_amount    month month_name   weekday quarter  
0        30       645.288  2024-11   November  Saturday  2024Q4  
1        15       108.511  2024-01    January   Tuesday  2024Q1  
2         0      1433.700  2024-05        May  Saturday  2024Q2  
3        30       346.206  2024-06       June    Sunday  2024Q2  
4        30        73.857  2024-10    October    Friday  2024Q4  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 14 columns):
 #   Column  

In [ ]:
from google.colab import drive
drive.mount('/content/drive')